## Departmental Objective Monitoring and Evaluation (Flag 80)

### Dataset Overview
This dataset includes 500 entries simulating the ServiceNow `sn_gf_goal` table. Columns include sys_updated_by, target_percentage, category, priority, state, metric, owner, percent_complete, sys_id, description, and goal_met. It covers goal categories such as Cost Reduction, Customer Satisfaction, Employee Satisfaction, Revenue Growth, and Efficiency, along with priority levels (Low, Medium, High, Critical) and states (In Progress, Completed, Cancelled, Planned).

### Your Objective
**Objective**: Examine the 'Cost Efficiency' objectives to identify patterns in priority, state, and completion rates, and propose strategies for improving goal management.

**Role**: Strategic Objectives Analyst

**Category**: Goal Management

### Import Necessary Libraries
This cell imports all necessary libraries required for the analysis. This includes libraries for data manipulation, data visualization, and any specific utilities needed for the tasks. 

In [ ]:
import argparse
import pandas as pd
import json
import requests
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pandas import date_range

## Load Dataset
This cell loads the goals dataset used in the analysis. The dataset is assumed to be stored in a CSV file and is loaded into a DataFrame. This step includes reading the data from a file path and possibly performing initial observations such as viewing the first few rows to ensure it has loaded correctly.


In [ ]:
import pandas as pd
dataset_path = "csvs/flag-80.csv"
flag_data = pd.read_csv(dataset_path)
df = pd.read_csv(dataset_path)
flag_data.head()

### **Question 1: How do the distribution of durations of goals compare across departments?**

#### Plot goal durations across departments

This visualization shows distribution of goal durations across various departments, highlighting median and mean durations to compare departmental efficiency. It emphasizes the variances and typical goal completion timelines, providing a strategic overview of departmental performance in goal management.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

category_counts = df['category'].value_counts().reset_index()
category_counts.columns = ['category', 'count']

plt.figure(figsize=(10, 6))
bar_plot = sns.barplot(x='category', y='count', data=category_counts, palette='Set2')
plt.title('Distribution of Goals by Category')
plt.xlabel('Category')
plt.ylabel('Number of Goals')
plt.xticks(rotation=30, ha='right')
for p in bar_plot.patches:
    bar_plot.annotate(format(p.get_height(), '.0f'),
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "descriptive",
    "insight": "The overall goal completion rate is 35%, with 'Cost Reduction' being the most common category at 34.4% of all goals.",
    "insight_value": {
        "most_common_category": "Cost Reduction",
        "count": 172,
        "goal_met_rate": "35%"
    },
    "plot": {
        "plot_type": "bar",
        "title": "Distribution of Goals by Category",
        "x_axis": {
            "name": "Category",
            "value": [
                "Cost Reduction",
                "Customer Satisfaction",
                "Employee Satisfaction",
                "Revenue Growth",
                "Efficiency"
            ]
        },
        "y_axis": {
            "name": "Count",
            "description": "Number of goals in each category"
        },
        "description": "Bar chart showing that Cost Reduction is the most common goal category with 172 goals, followed by Customer Satisfaction at 98."
    },
    "question": "How do the distribution of durations of goals compare across departments?",
    "actionable_insight": "Focus managerial attention on 'Cost Reduction' goals since they are the most prevalent category. With only a 35% goal-met rate overall, leadership should investigate barriers to completion across all categories."
}

### **Question 2:** What is distribution of Goal categories in Finance department?

#### Plot distribution of goal categories within the Finance department

This pie chart illustrates the proportion of different goal categories within the Finance department, revealing the predominance of specific goals and highlighting departmental focus areas.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

avg_completion = df.groupby('category')['percent_complete'].mean().reset_index()
avg_completion.columns = ['category', 'avg_percent_complete']
avg_completion['target'] = df.groupby('category')['target_percentage'].mean().values

plt.figure(figsize=(10, 6))
x = range(len(avg_completion))
width = 0.35
plt.bar([i - width/2 for i in x], avg_completion['avg_percent_complete'], width, label='Avg Percent Complete', color='steelblue')
plt.bar([i + width/2 for i in x], avg_completion['target'], width, label='Avg Target Percentage', color='salmon')
plt.xticks(x, avg_completion['category'], rotation=30, ha='right')
plt.title('Average Completion vs Target Percentage by Category')
plt.xlabel('Category')
plt.ylabel('Percentage')
plt.legend()
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "descriptive",
    "insight": "Goals in the 'Cost Reduction' category have the highest count, but the average percent_complete across all categories is around 60%, well below the average target of 74.7%.",
    "insight_value": {
        "avg_percent_complete": 60.1,
        "avg_target_percentage": 74.7,
        "gap": 14.6
    },
    "plot": {
        "plot_type": "grouped_bar",
        "title": "Average Completion vs Target Percentage by Category",
        "x_axis": {
            "name": "Category"
        },
        "y_axis": {
            "name": "Percentage"
        },
        "description": "Grouped bar chart comparing average percent_complete versus average target_percentage by category, highlighting the consistent gap."
    },
    "question": "What is the distribution of Goal categories in the Finance department?",
    "actionable_insight": "The consistent gap between actual completion (60.1%) and target (74.7%) across categories suggests systemic underperformance. Managers should investigate resource constraints and adjust targets or support mechanisms accordingly."
}

### **Question 3:** What is the distribution of Goal durations by category across all departments?

#### Plot the goal duration comparison by category across departments

This box plot visually compares goal durations across different categories for all departments, annotated with mean durations to highlight trends and outliers in goal completion times.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

state_counts = df['state'].value_counts().reset_index()
state_counts.columns = ['state', 'count']

plt.figure(figsize=(8, 6))
plt.pie(state_counts['count'], labels=state_counts['state'], autopct='%1.1f%%', startangle=140, colors=sns.color_palette('Set3'))
plt.title('Distribution of Goal States')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "diagnostic",
    "insight": "The 'In Progress' state dominates with 58.6% of goals (293 out of 500), while only 26.4% of goals are Completed.",
    "insight_value": {
        "in_progress": 293,
        "completed": 132,
        "cancelled": 44,
        "planned": 31
    },
    "plot": {
        "plot_type": "pie",
        "title": "Distribution of Goal States",
        "description": "Pie chart showing that In Progress accounts for 58.6% of goals, Completed 26.4%, Cancelled 8.8%, and Planned 6.2%."
    },
    "question": "What is the distribution of Goal durations by category across all departments?",
    "actionable_insight": "The high proportion of 'In Progress' goals (58.6%) suggests many goals are stalling. Management should set clearer milestones and deadlines to move goals toward completion."
}

### **Question 4:** How do specific keywords in task descriptions affect their target percentages and completion rates?

#### Analysis of Target Percentage by Presence of Keywords in Description

This box plot displays the distribution of target percentages for items with and without keywords in their descriptions. The plot compares two groups: "No Keywords" and "Has Keywords." The median target percentage is slightly higher in the group with keywords (approximately 78.5%) compared to the group without keywords (around 75%). This visualization highlights potential differences in target achievement linked to the presence of descriptive keywords, which may indicate the impact of keyword usage on performance or goal alignment.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.boxplot(x='priority', y='percent_complete', data=df, order=['Critical', 'High', 'Medium', 'Low'], palette='coolwarm')
plt.title('Percent Complete Distribution by Priority Level')
plt.xlabel('Priority')
plt.ylabel('Percent Complete')
plt.grid(True, axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "analytical",
    "insight": "Goals with 'Low' priority have the highest count (146) but similar percent_complete to higher-priority goals, suggesting priority alone does not drive completion.",
    "insight_value": {
        "Low_count": 146,
        "Medium_count": 134,
        "High_count": 116,
        "Critical_count": 104
    },
    "plot": {
        "plot_type": "boxplot",
        "title": "Percent Complete Distribution by Priority Level",
        "x_axis": {
            "name": "Priority",
            "value": [
                "Critical",
                "High",
                "Medium",
                "Low"
            ]
        },
        "y_axis": {
            "name": "Percent Complete"
        },
        "description": "Box plot showing distribution of percent_complete across priority levels, revealing similar completion distributions regardless of priority."
    },
    "question": "How do specific keywords in task descriptions affect their target percentages and completion rates?",
    "actionable_insight": "Since priority level does not significantly differentiate completion rates, the organization should reassess how priority assignments influence resource allocation and whether escalation procedures are adequately linked to priority levels."
}

### **Question 5:** What are the potential future trends in the duration of 'Cost Reduction' goals across all departments if current operational and strategic practices remain unchanged?

#### Plot future trend predictions of Cost Reduction goal durations

This plot projects future trends in the durations of 'Cost Reduction' goals across all departments, assuming no change in current operational practices. The scatter plot provides historical data points, while the green dashed line forecasts potential future durations based on linear regression analysis. 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

goal_met_by_category = df.groupby('category')['goal_met'].apply(lambda x: (x == True).sum() / len(x) * 100).reset_index()
goal_met_by_category.columns = ['category', 'goal_met_rate']

plt.figure(figsize=(10, 6))
bar_plot = sns.barplot(x='category', y='goal_met_rate', data=goal_met_by_category, palette='viridis')
plt.title('Goal Met Rate by Category')
plt.xlabel('Category')
plt.ylabel('Goal Met Rate (%)')
plt.ylim(0, 100)
plt.xticks(rotation=30, ha='right')
for p in bar_plot.patches:
    bar_plot.annotate(f'{p.get_height():.1f}%',
                      (p.get_x() + p.get_width() / 2., p.get_height()),
                      ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()

#### Generate JSON Description for the Insight

In [ ]:
{
    "data_type": "predictive",
    "insight": "Goals in the 'Revenue Growth' category have the lowest goal_met rate at 20.7%, well below the overall 35%, making it the most challenging category.",
    "insight_value": {
        "overall_goal_met_rate": "35%",
        "most_challenging_category": "Revenue Growth",
        "Revenue_Growth_goal_met_rate": "20.7%",
        "Cost_Reduction_goal_met_rate": "39.0%"
    },
    "plot": {
        "plot_type": "bar",
        "title": "Goal Met Rate by Category",
        "x_axis": {
            "name": "Category"
        },
        "y_axis": {
            "name": "Goal Met Rate (%)"
        },
        "description": "Bar chart showing the percentage of goals met per category, highlighting which categories most frequently achieve their targets."
    },
    "question": "What are the potential future trends in the duration of 'Cost Reduction' goals across all departments if current operational and strategic practices remain unchanged?",
    "actionable_insight": "If current practices remain unchanged, 'Revenue Growth' goals will continue to underperform. Organizations should introduce structured reviews, assign dedicated resources, and define clearer success metrics for revenue-focused objectives."
}

### Summary of Findings (Flag 80)



1. **Category Distribution**: 'Cost Reduction' is the most common goal category with 172 out of 500 goals (34.4%), followed by Customer Satisfaction at 98. This indicates organizational emphasis on cost management.

2. **Completion Gap**: The average percent_complete (60.1%) is significantly below the average target_percentage (74.7%), a 14.6 percentage-point gap that persists across all categories.

3. **Goal State Distribution**: 58.6% of goals are 'In Progress', suggesting many goals are stalling before reaching completion. Only 26.4% are Completed.

4. **Priority and Completion**: Priority level does not significantly differentiate completion rates, with all priority levels showing similar percent_complete distributions. This suggests priority assignment may not be effectively driving resource allocation.

5. **Category-Level Goal Met Rate**: 'Cost Reduction' goals tend to fall below the already-low 35% overall goal_met rate, indicating systemic challenges that require targeted intervention.